# California Counties Ranked by Berry Acreage (2022)

Top-10 California counties by **acres grown** for strawberries and for all berries combined, using USDA NASS 2022 Census of Agriculture county data (most recent census)

Using the NASS Quickstats API 

Source files:
- `data/strawberry_acres_2022_CAcounties.csv` — strawberry acres grown
- `data/berry_totals_2022_CAcounties.csv` — all-berry acres grown

Values reported as `(D)` are withheld by USDA to avoid disclosing individual operations and are dropped.

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import requests
from dotenv import load_dotenv

load_dotenv()
api_key = os.environ["NASS_API_KEY"]

## Live NASS Quick Stats query: strawberry area grown by CA county

Most relevant to plastic mulch use

In [10]:
NASS_BASE_URL = "https://quickstats.nass.usda.gov/api/api_GET/"

def nass_get(params):
    params = {**params, "key": NASS_API_KEY, "format": "JSON"}
    resp = requests.get(NASS_BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json().get("data", [])

def most_recent_year(params):
    years_url = "https://quickstats.nass.usda.gov/api/get_param_values/"
    resp = requests.get(years_url, params={**params, "key": NASS_API_KEY, "param": "year"}, timeout=30)
    resp.raise_for_status()
    years = resp.json().get("year", [])
    return max((int(y) for y in years), default=None)

base_query = {"commodity_desc": "STRAWBERRIES", "state_alpha": "CA", "agg_level_desc": "COUNTY"}

# AREA GROWN (the modern name for AREA HARVESTED) is the most recent county-level metric
county_stat = "AREA GROWN"
latest_year = most_recent_year({**base_query, "statisticcat_desc": county_stat})

records = nass_get(
    {
        **base_query,
        "statisticcat_desc": county_stat,
        "year": latest_year,
        "domain_desc": "TOTAL",
        "short_desc": f"STRAWBERRIES - ACRES GROWN",
    }
)

nass_county = pd.DataFrame(records)
nass_county["Value"] = pd.to_numeric(nass_county["Value"].str.replace(",", ""), errors="coerce")
nass_county = nass_county.dropna(subset=["Value"])
nass_county["County"] = nass_county["county_name"].str.title()
nass_county = nass_county[["County", "Value"]].sort_values("Value", ascending=False)

print(f"Most recent year with CA county-level strawberry data: {latest_year} (statistic: {county_stat})")
nass_county.head(10)


Most recent year with CA county-level strawberry data: 2022 (statistic: AREA GROWN)


,County,Value
39,Santa Barbara,15684.0
9,Monterey,13586.0
40,Ventura,12027.0
12,San Luis Obispo,3099.0
15,Santa Cruz,1797.0
35,Orange,189.0
10,Napa,90.0
23,Kern,62.0
36,Riverside,55.0
38,San Diego,51.0


## Live NASS Quick Stats query: total berries (all types) area grown by CA county (2022)

In [7]:
berry_totals_records = nass_get(
    {
        "commodity_desc": "BERRY TOTALS",
        "state_alpha": "CA",
        "agg_level_desc": "COUNTY",
        "statisticcat_desc": "AREA GROWN",
        "year": 2022,
        "domain_desc": "TOTAL",
        "short_desc": "BERRY TOTALS - ACRES GROWN",
    }
)

berry_totals_county = pd.DataFrame(berry_totals_records)
berry_totals_county["Value"] = pd.to_numeric(berry_totals_county["Value"].str.replace(",", ""), errors="coerce")
berry_totals_county = berry_totals_county.dropna(subset=["Value"])
berry_totals_county["County"] = berry_totals_county["county_name"].str.title()
berry_totals_county = berry_totals_county[["County", "Value"]].sort_values("Value", ascending=False)

berry_totals_county.head(10)


,County,Value
52,Santa Barbara,18548.0
53,Ventura,17443.0
13,Monterey,14368.0
19,Santa Cruz,3786.0
16,San Luis Obispo,3295.0
37,Tulare,2765.0
31,Kern,2177.0
35,San Joaquin,2106.0
30,Fresno,1141.0
51,San Diego,445.0


## Live NASS Quick Stats query: all individual berry types, area grown by CA county (2022)

Sums `AREA GROWN` (acres) across every individual berry commodity (name contains "BERRY"/"BERRIES"), excluding the aggregate `BERRY TOTALS` commodity, then ranks counties by the combined total.

Results in slightly different results, I trust the berry totals more 

In [4]:
berry_commodities_url = "https://quickstats.nass.usda.gov/api/get_param_values/"
resp = requests.get(
    berry_commodities_url,
    params={
        "key": NASS_API_KEY,
        "param": "commodity_desc",
        "state_alpha": "CA",
        "agg_level_desc": "COUNTY",
        "year": 2022,
        "statisticcat_desc": "AREA GROWN",
    },
    timeout=30,
)
resp.raise_for_status()
all_commodities = resp.json().get("commodity_desc", [])
berry_commodities = [
    c for c in all_commodities if ("BERRY" in c or "BERRIES" in c) and c != "BERRY TOTALS"
]

berry_records = []
for commodity in berry_commodities:
    berry_records.extend(
        nass_get(
            {
                "commodity_desc": commodity,
                "state_alpha": "CA",
                "agg_level_desc": "COUNTY",
                "statisticcat_desc": "AREA GROWN",
                "year": 2022,
                "domain_desc": "TOTAL",
            }
        )
    )

berry_all = pd.DataFrame(berry_records)
# short_desc naming varies by commodity (e.g. "BLACKBERRIES, INCL DEWBERRIES...- ACRES GROWN"),
# so filter on the "ACRES GROWN" suffix rather than the "OPERATIONS WITH AREA GROWN" counts.
berry_all = berry_all[berry_all["short_desc"].str.endswith("ACRES GROWN")]
berry_all["Value"] = pd.to_numeric(berry_all["Value"].str.replace(",", ""), errors="coerce")
berry_all = berry_all.dropna(subset=["Value"])
berry_all["County"] = berry_all["county_name"].str.title()

berry_by_county = (
    berry_all.groupby("County")["Value"].sum().rename("Value").reset_index().sort_values("Value", ascending=False)
)

print(f"Berry commodities included: {', '.join(berry_commodities)}")
berry_by_county.head(10)


Berry commodities included: ARONIA BERRIES, BERRIES, OTHER, BLACKBERRIES, BLUEBERRIES, BOYSENBERRIES, ELDERBERRIES, GOOSEBERRIES, LOGANBERRIES, MULBERRIES, RASPBERRIES, STRAWBERRIES


,County,Value
27,Santa Barbara,18680.0
36,Ventura,18321.0
14,Monterey,14400.0
34,Tulare,5442.0
7,Kern,4276.0
24,San Joaquin,4134.0
28,Santa Cruz,3904.0
25,San Luis Obispo,3346.0
4,Fresno,2212.0
23,San Diego,786.0


## Live NASS Quick Stats query: blackberries area grown by CA county (2022)

In [5]:
def area_grown_by_county(commodity_desc):
    records = nass_get(
        {
            "commodity_desc": commodity_desc,
            "state_alpha": "CA",
            "agg_level_desc": "COUNTY",
            "statisticcat_desc": "AREA GROWN",
            "year": 2022,
            "domain_desc": "TOTAL",
        }
    )
    df = pd.DataFrame(records)
    # short_desc naming varies by commodity, so filter on the "ACRES GROWN" suffix.
    df = df[df["short_desc"].str.endswith("ACRES GROWN")]
    df["Value"] = pd.to_numeric(df["Value"].str.replace(",", ""), errors="coerce")
    df = df.dropna(subset=["Value"])
    df["County"] = df["county_name"].str.title()
    return df.groupby("County")["Value"].sum().rename("Value").reset_index().sort_values("Value", ascending=False)


blackberry_by_county = area_grown_by_county("BLACKBERRIES")
blackberry_by_county.head(10)


,County,Value
24,Ventura,1079.0
17,Santa Cruz,974.0
8,Monterey,438.0
20,Sonoma,140.0
2,Fresno,68.0
9,Napa,64.0
22,Tulare,59.0
1,El Dorado,44.0
18,Shasta,38.0
16,San Mateo,31.0


## Live NASS Quick Stats query: raspberries area grown by CA county (2022)

Most relevant to hoop houses

In [6]:
raspberry_by_county = area_grown_by_county("RASPBERRIES")
raspberry_by_county.head(10)


,County,Value
17,Ventura,3459.0
13,Santa Barbara,1308.0
14,Santa Cruz,889.0
6,Monterey,309.0
12,San Luis Obispo,106.0
15,Sonoma,32.0
1,El Dorado,24.0
10,San Bernardino,8.0
9,Riverside,8.0
11,San Diego,6.0
